# Gold Store Performance

## Purpose
Create daily store-level business KPIs from Silver transactions.

### Source
retailanalytics.silver.silver_transaction

### Target
retailanalytics.gold.store_performance

### Grain
sale_date + store_id

### KPIs
- Total Revenue
- Completed Transactions
- Failed Transactions
- Pending Transactions
- Total Units Sold

In [0]:
from pyspark.sql import functions as F

In [0]:
store_checkpoint = (
    "/Volumes/retailanalytics/secrets/"
    "kafkacerts/checkpoints/gold_store_performance"
)

In [0]:
silver_stream = (
    spark.readStream.table(
        "retailanalytics.silver.silver_transaction"
    )
)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS retailanalytics.gold.store_performance
(
    sale_date DATE,
    store_id STRING,
    total_revenue DOUBLE,
    completed_transactions BIGINT,
    failed_transactions BIGINT,
    pending_transactions BIGINT,
    total_units BIGINT
)
USING DELTA;

In [0]:
def upsert_store_performance(batch_df, batch_id):

    store_batch = (
        batch_df
        .groupBy(
            F.to_date("transaction_ts").alias("sale_date"),
            F.col("store_id")
        )
        .agg(
            F.sum(
                F.when(
                    F.col("transaction_status") == "COMPLETED",
                    F.col("gross_amount")
                ).otherwise(0)
            ).alias("total_revenue"),

            F.sum(
                F.when(
                    F.col("transaction_status") == "COMPLETED",
                    1
                ).otherwise(0)
            ).alias("completed_transactions"),

            F.sum(
                F.when(
                    F.col("transaction_status") == "FAILED",
                    1
                ).otherwise(0)
            ).alias("failed_transactions"),

            F.sum(
                F.when(
                    F.col("transaction_status") == "PENDING",
                    1
                ).otherwise(0)
            ).alias("pending_transactions"),

            F.sum(
                F.when(
                    F.col("transaction_status") == "COMPLETED",
                    F.col("quantity")
                ).otherwise(0)
            ).alias("total_units")
        )
    )

    store_batch.createOrReplaceTempView(
        "store_performance_updates"
    )

    spark.sql("""
        MERGE INTO retailanalytics.gold.store_performance AS target
        USING store_performance_updates AS source

        ON target.sale_date = source.sale_date
        AND target.store_id = source.store_id

        WHEN MATCHED THEN
          UPDATE SET
            target.total_revenue =
                target.total_revenue + source.total_revenue,

            target.completed_transactions =
                target.completed_transactions + source.completed_transactions,

            target.failed_transactions =
                target.failed_transactions + source.failed_transactions,

            target.pending_transactions =
                target.pending_transactions + source.pending_transactions,

            target.total_units =
                target.total_units + source.total_units

        WHEN NOT MATCHED THEN
          INSERT (
              sale_date,
              store_id,
              total_revenue,
              completed_transactions,
              failed_transactions,
              pending_transactions,
              total_units
          )
          VALUES (
              source.sale_date,
              source.store_id,
              source.total_revenue,
              source.completed_transactions,
              source.failed_transactions,
              source.pending_transactions,
              source.total_units
          )
    """)

In [0]:
(
    silver_stream.writeStream
        .foreachBatch(upsert_store_performance)
        .option(
            "checkpointLocation",
            store_checkpoint
        )
        .trigger(availableNow=True)
        .start()
)

In [0]:
%sql
SELECT
    sale_date,
    SUM(total_revenue) AS store_revenue
FROM retailanalytics.gold.store_performance
GROUP BY sale_date
ORDER BY sale_date;

In [0]:
%sql
SELECT
    sale_date,
    total_revenue
FROM retailanalytics.gold.daily_sales
ORDER BY sale_date;

--19270564.919999998
--31016500.889999997